# Horizon on time series, take two: with enough data not to memorise

**What went wrong last time.** The same measurement on four time-series corpora produced
alpha values that looked interesting and meant nothing. Every run had validation loss
*above* the uniform baseline of 5.545 nats, and the giveaway was this:

```
AR_phi0.00 (WHITE NOISE):  train loss 5.697 -> 5.549 -> 5.381 -> 2.015
```

You cannot learn white noise. Training loss belongs at 5.545 forever. Reaching 2.015 means
the model memorised the specific training sequence.

**The cause was tokens per parameter.**

| run | tokens | params | tokens/param | overfit? |
|---|---|---|---|---|
| enwik8 | 18,000,000 | 11.9M | **1.5** | no |
| AR (last attempt) | 360,000 | 11.9M | **0.03** | yes, catastrophically |

Memorisation is a capacity threshold, not a slide: the model sat at the uniform baseline for
3000 steps, then found the shortcut and collapsed by 4500.

**Why not just shrink the model?** Matching enwik8's ratio at 360k tokens needs ~240k
parameters, i.e. around dim 32-48. That is the regime this project's main result shows is
misleading -- alpha is 0.741 at dim 64 against 0.227 at dim 384. Shrinking the model would
confound any corpus effect with the width effect we are trying to hold fixed.

**So the data grows instead.** AR is synthetic, so 20M points is free. `electricity_15min`
has ~370 series of ~105k points, roughly 39M available.

**And this time the run asserts.** Validation loss is checked mid-training, not just at the
end. A corpus that is memorising aborts instead of producing plausible numbers.

## 1 - Setup

In [ ]:
!pip install -q titans-pytorch datasets
!git clone -q https://github.com/thebnbrkr/marv-titan.git /content/marv-titan 2>/dev/null || true
!git clone -q --depth 1 https://github.com/lucidrains/titans-pytorch.git /content/titans-src 2>/dev/null || true
import sys; sys.path.insert(0, '/content/marv-titan/experiments')

import torch, numpy as np, time, json, os
from titans_horizon_timeseries import (quantize, marginal_entropy, build,
                                       sample_batch, measure_alpha, val_loss, N_BINS)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
UNIFORM = float(np.log(N_BINS))
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE=="cuda" else "")
print(f"uniform baseline = {UNIFORM:.3f} nats -- any val loss above this means failure")

In [ ]:
DIM      = 384          # held fixed: width is the variable the main result shows matters
STEPS    = 6000
SEQ_LEN  = 256
BATCH    = 8
LR       = 2e-4
SEG      = 8
PASSAGE  = 1024
SEED     = 0
TOKEN_TARGET = 20_000_000       # was 400_000 -- that was the whole problem

DRAWS = STEPS * BATCH * SEQ_LEN
print(f"token draws over training: {DRAWS:,}")
print(f"target corpus size       : {TOKEN_TARGET:,}  ->  {DRAWS/TOKEN_TARGET:.2f} epochs")
print(f"enwik8 reference         : 18,000,000 tokens, ~2.8 epochs, did NOT overfit")

# reference points from the enwik8 runs
REF = {"dim64_enwik8": dict(alpha=0.741, tau_pos=5.9, val=1.974),
       "dim384_enwik8": dict(alpha=0.227, tau_pos=31.1, val=1.248)}

## 2 - Corpora at scale

AR(1) via `lfilter`, which is the same recursion `x_t = phi*x_{t-1} + e_t` computed in O(n)
instead of a Python loop -- 20M points in seconds rather than minutes.

`electricity_15min` streams until the token budget is met. It is the one real corpus in the
list with enough data; ETT tops out around 500k points and cannot reach this scale, which is
why it is not here.

In [ ]:
def ar1_big(n, phi, seed=0):
    """AR(1) at scale. lfilter computes the recursion directly; the loop fallback
    matches it exactly but is far slower."""
    rng = np.random.default_rng(seed)
    e = rng.standard_normal(n)
    if phi == 0.0:
        return e                      # white noise, no recursion needed
    try:
        from scipy.signal import lfilter
        return lfilter([1.0], [1.0, -phi], e)
    except ImportError:
        x = np.zeros(n)
        for t in range(1, n): x[t] = phi*x[t-1] + e[t]
        return x


def stream_chronos(config, target_tokens, repo="autogluon/chronos_datasets"):
    """Stream series until the token budget is met."""
    from datasets import load_dataset
    ds = load_dataset(repo, config, split="train", streaming=True)
    parts, total, col = [], 0, None
    for row in ds:
        if col is None:
            best, bl = None, 0
            for k, v in row.items():
                if isinstance(v, (list, np.ndarray)) and len(v) > bl:
                    try: float(v[0]); best, bl = k, len(v)
                    except (TypeError, ValueError, IndexError): pass
            col = best
            if col is None: raise ValueError(f"no numeric sequence column in {config}")
            print(f"    column '{col}'")
        s = np.asarray(row[col], dtype=np.float64); s = s[np.isfinite(s)]
        if len(s) < 200: continue
        parts.append(s); total += len(s)
        if total >= target_tokens: break
    print(f"    {len(parts)} series -> {total:,} points")
    return np.concatenate(parts)


CORPORA = {
    "AR_phi0.00": lambda: ar1_big(TOKEN_TARGET, 0.00, SEED),   # white noise -- the control
    "AR_phi0.90": lambda: ar1_big(TOKEN_TARGET, 0.90, SEED),   # predictable
    "electricity_15min": lambda: stream_chronos("electricity_15min", TOKEN_TARGET),
}

## 3 - Pre-flight

Three things must hold before any GPU time is spent: marginal entropy pinned near 5.545,
enough tokens that memorisation is not available, and lag-1 structure that actually differs
between the AR pair.

In [ ]:
def lag1_mi(tok, coarse=16):
    x = (tok.numpy().astype(int) * coarse) // N_BINS
    J = np.histogram2d(x[:-1], x[1:], bins=[coarse, coarse])[0]; J = J/J.sum()
    px, py = J.sum(1, keepdims=True), J.sum(0, keepdims=True); m = J > 0
    return float((J[m]*np.log(J[m]/(px@py)[m])).sum())

prepared = {}
n_params = sum(p.numel() for p in build(DIM).parameters())
print(f"model parameters: {n_params/1e6:.1f}M\n")
print(f"{'corpus':<20}{'train tokens':>14}{'tok/param':>11}{'entropy':>9}{'lag-1 MI':>10}")
for name, loader in CORPORA.items():
    t0 = time.time()
    raw = loader()
    tr, va = quantize(raw)
    prepared[name] = (tr, va)
    ratio = len(tr)/n_params
    print(f"{name:<20}{len(tr):>14,}{ratio:>11.2f}{marginal_entropy(tr):>9.3f}"
          f"{lag1_mi(tr):>10.4f}   ({time.time()-t0:.0f}s)")
print(f"\nenwik8 reference: 18,000,000 tokens, 1.51 tok/param, no overfitting")
print("a ratio near or above 1 is what we are buying here; 0.03 is what broke last time")

## 4 - Train, with a mid-run abort

The abort triggers on the **train-validation gap**, not on validation loss alone.
Memorisation shows up as training loss far below validation loss -- last attempt's white
noise hit train 2.015 against val 9.743, a gap of 7.7. A model sitting at
train ~= val ~= 5.545 on white noise is *correct*, not broken: uniform is the best
achievable loss on an unlearnable corpus.

In [ ]:
RESULTS = "horizon_v2.json"
rows = json.load(open(RESULTS)) if os.path.exists(RESULTS) else []
done = {r["corpus"] for r in rows}

for name, (tr, va) in prepared.items():
    if name in done:
        print(f"skip {name} (done)"); continue
    torch.manual_seed(SEED); np.random.seed(SEED)
    model = build(DIM).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    passage = va[:PASSAGE]
    t0 = time.time(); aborted = False

    model.train()
    for step in range(STEPS):
        loss = model(sample_batch(tr, SEQ_LEN, BATCH).to(DEVICE), return_loss=True)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        opt.step()

        if step in (1500, 3000, 4500):
            v = val_loss(model, va, SEQ_LEN, BATCH, DEVICE, n=10)
            gap = v - loss.item()
            print(f"    {name} step {step:>5}  train {loss.item():.3f}  val {v:.3f}"
                  f"  gap {gap:+.3f}", flush=True)
            # Memorisation shows as train FAR BELOW val, not as val near uniform.
            # White noise correctly sits at train ~= val ~= uniform (5.545): that is the
            # right answer for an unlearnable corpus, not a failure.
            if gap > 1.0:
                print(f"    !! ABORT {name}: train {loss.item():.3f} vs val {v:.3f} "
                      f"(gap {gap:.2f}) -- memorising the corpus\n", flush=True)
                aborted = True; break

    if aborted:
        rows.append({"corpus": name, "status": "aborted_overfit"})
        json.dump(rows, open(RESULTS,"w"), indent=2)
        del model, opt
        if DEVICE=="cuda": torch.cuda.empty_cache()
        continue

    a = measure_alpha(model, passage, DEVICE); tau = -1.0/np.log(1.0-a)
    v = val_loss(model, va, SEQ_LEN, BATCH, DEVICE)
    row = {"corpus": name, "status": "ok", "alpha": a, "tau_chunks": float(tau),
           "tau_positions": float(tau*SEG), "pct_context": float(tau*SEG/SEQ_LEN*100),
           "val_loss": v, "train_tokens": int(len(tr)),
           "marginal_entropy": marginal_entropy(tr), "minutes": (time.time()-t0)/60}
    rows.append(row); json.dump(rows, open(RESULTS,"w"), indent=2)
    print(f"  -> {name}: alpha {a:.4f} | tau {row['tau_positions']:.1f} pos "
          f"({row['pct_context']:.1f}%) | val {v:.3f} | {row['minutes']:.0f} min\n", flush=True)
    del model, opt
    if DEVICE=="cuda": torch.cuda.empty_cache()

import pandas as pd
pd.DataFrame(rows).round(4)

## 5 - Compare against the text runs

In [ ]:
ok = [r for r in rows if r.get("status") == "ok"]
if not ok:
    print("no corpus completed -- every run aborted on the overfitting check.")
else:
    print(f"{'corpus':<20}{'alpha':>8}{'tau (pos)':>11}{'% context':>11}{'val':>8}")
    for r in ok:
        print(f"{r['corpus']:<20}{r['alpha']:>8.4f}{r['tau_positions']:>11.1f}"
              f"{r['pct_context']:>10.1f}%{r['val_loss']:>8.3f}")
    print(f"{'enwik8 (text)':<20}{REF['dim384_enwik8']['alpha']:>8.3f}"
          f"{REF['dim384_enwik8']['tau_pos']:>11.1f}{12.1:>10.1f}%{REF['dim384_enwik8']['val']:>8.3f}")

    a = [r["alpha"] for r in ok] + [REF["dim384_enwik8"]["alpha"]]
    print(f"\nalpha spread across corpora (all dim {DIM}): {min(a):.3f} - {max(a):.3f}"
          f"  ({max(a)/min(a):.1f}x)")
    print(f"the WIDTH effect, for comparison:                0.227 - 0.741  (3.3x)")
    print("\nif the corpus spread is small next to 3.3x, the horizon is set by")
    print("architecture and width, and measuring it on text was legitimate.")
    ar = {r["corpus"]: r["alpha"] for r in ok if r["corpus"].startswith("AR_")}
    if len(ar) == 2:
        print(f"\ncontrolled pair (identical size and marginal, only temporal dependence differs):")
        for k in sorted(ar): print(f"   {k}: alpha {ar[k]:.4f}")

## How to read it

**If corpora abort**, the token budget is still too small -- raise `TOKEN_TARGET` and rerun.
The abort is the experiment working, not failing.

**The AR pair is the claim.** Identical size, identical marginal entropy, differing only in
temporal dependence. If alpha moves between them, temporal structure is doing it. If it does
not, the horizon is architectural.

**Compare any spread against 3.3x**, the width effect measured over five checkpoints. A
corpus spread much smaller than that means the enwik8 measurement transfers.

**One seed.** The width effect survived five checkpoints across a 25x training range; a
corpus effect deserves the same scepticism. Treat a small difference as noise.